In [ ]:
import matlab.engine
import os
import torch
import json
from mylib import *
import math
import numpy as np
from pycocotools.coco import COCO
import cv2
import matplotlib.pyplot as plt
import skimage.io as io
from ultralytics import YOLO 
from dotenv import load_dotenv
from tqdm import tqdm  # For progress tracking
import time
load_dotenv()  # This loads from .env in the current directory

In [ ]:
# Check GPU availability
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.memory_allocated(0)/1024**2, 2), "MB")
    print("Memory Reserved:", round(torch.cuda.memory_reserved(0)/1024**2, 2), "MB")
    print("Total Memory:", round(torch.cuda.get_device_properties(0).total_memory/1024**2, 2), "MB")
else:
    print("No CUDA-compatible GPU detected.")

In [ ]:
# Load task classes
with open('indoor_objects.json') as f:
    indoor_data = json.load(f)
    print("Indoor data:", indoor_data)

# Load annotations for both datasets
cocos = {
    "train2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_train2017.json")),
    "val2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_val2017.json")),
}

# Load COCO categories 
train_coco = list(cocos.values())[0]
all_cats = train_coco.loadCats(train_coco.getCatIds())

# Build mapping from YOLO IDs to COCO IDs (YOLO goes from 0 to 79, COCO goes from 1 to 90 and skips some IDs, but order is the same)
yolo_id_to_coco_id = {i: cat["id"] for i, cat in enumerate(sorted(all_cats, key=lambda x: x["id"]))} # eg. Yolo ID 79 -> COCO ID 90
coco_id_to_name = {cat["id"]: cat["name"] for cat in all_cats} # eg. COCO ID 90 -> "toothbrush"
coco_classes_ids = list(coco_id_to_name.keys()) # eg. [1, 2, 3, ..., 90]

# Print mappings
print("Yolo ID to COCO ID mapping:", yolo_id_to_coco_id)
print("COCO ID to name mapping:", coco_id_to_name)
print("COCO classes IDs mapping:", coco_classes_ids)

# Load bins for each class
with open("out_bins_per_class.json") as f:
    out_bins_per_class = json.load(f)

# Count number of classes 
N = len(out_bins_per_class)

# Print information
print("Number of classes:", N)

In [ ]:
# === Load YOLO model ===
yolo_model = YOLO("yolov8n.pt")  # Swap with yolov8s.pt or yolov8x.pt as needed

In [ ]:
# === Score storage: {class_name: {bin_index: [scores]}} ===
score_vectors_by_class_bin = {
    class_name: {i: [] for i in range(len(out_bins_per_class[class_name]) - 1)}
    for class_name in out_bins_per_class
}

# IoU threshold
iou_threshold = 0.5

# === Helper: Get COCO split for a given image ID ===
def find_split_for_image(img_id):
    for split, coco_obj in cocos.items():
        if img_id in coco_obj.imgs:
            return split, coco_obj
    return None, None

In [ ]:
# Loop through every class_id
for class_id in coco_classes_ids:
    # Get the corresponding class name
    class_name = coco_id_to_name[class_id]

    # Get all image IDs for this class (from both splits)
    img_ids = []
    for split, coco_obj in cocos.items():
        img_ids += coco_obj.getImgIds(catIds=[class_id])

    # Loop through each image ID
    for img_id in tqdm(img_ids, desc=f"Processing {len(img_ids)} images for {class_name}", unit="image"):
        
        # Find the split for the current image ID
        split, any_coco = find_split_for_image(img_id)
        
        # Load the image metadata
        img_data = any_coco.imgs[img_id]
        img_path = os.path.join(os.getenv("COCO_DATA"), split, img_data["file_name"])

        # Fix image color and load failure
        img = cv2.imread(img_path)

        # Use predict() explicitly
        results = yolo_model.predict(img, device='cuda:0', conf=0.01, iou=0.5, verbose=False)

        print(results)

# TODO: GET CONFIDENCE SCORES, COMPARE GROUND WITH PREDICTED USING IOU, ASSIGN CONFIDENCE SCORES TO BINS

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.memory_allocated(0)/1024**2, 2), "MB")
    print("Memory Reserved:", round(torch.cuda.memory_reserved(0)/1024**2, 2), "MB")
    print("Total Memory:", round(torch.cuda.get_device_properties(0).total_memory/1024**2, 2), "MB")
else:
    print("No CUDA-compatible GPU detected.")

In [ ]:
# Function to print RAM usage
def print_ram_usage():
    import psutil
    ram = psutil.virtual_memory()
    print(f"RAM usage: {ram.percent}% used, {ram.available / (1024 ** 3):.2f} GB available")
    print(f"RAM total: {ram.total / (1024 ** 3):.2f} GB total")
    print(f"RAM used: {ram.used / (1024 ** 3):.2f} GB used")
    print(f"RAM free: {ram.free / (1024 ** 3):.2f} GB free")

In [ ]:
eng = matlab.engine.start_matlab()
eng.addpath(os.getenv("FASTFIT_TOOLBOX"), nargout=0)

eng.quit()